<a href="https://colab.research.google.com/github/kuds/rl-doom/blob/main/notebooks/06_dreamer_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — DreamerV3 World-Model Training

Train a **DreamerV3** agent (via the `NM512/dreamerv3-torch` PyTorch port,
wrapped by `rl_doom.agents.dreamer.train_dreamer`) on four ViZDoom
scenarios: *Basic*, *Deadly Corridor*, *Defend the Center*, and *Deathmatch*.

**Why a world model?** Unlike PPO/DQN (model-free) the agent here learns
an internal RSSM that predicts the next latent + reward. The actor and
critic are then trained on imagined rollouts in latent space, so each
real env step is amortised over many gradient updates. This trades extra
compute per step for dramatically better sample efficiency — a good fit
for ViZDoom, where each env step drives a native game process.

**Config-driven.** All hyperparameters, env settings, and training budgets
come from `configs/dreamer_<scenario>.yaml` so the notebook and the YAML
files can never drift apart — same pattern as the PPO/DQN notebooks.

**Upstream port.** The notebook `git clone`s `NM512/dreamerv3-torch`
pinned to `rl_doom.agents.dreamer.UPSTREAM_PIN` and adds it to `sys.path`.
Our wrapper imports the four files we actually need (`models.py`,
`networks.py`, `tools.py`, `exploration.py`) and re-implements the thin
`Dreamer` driver inline so we avoid the upstream's `gym` + `ruamel.yaml`
deps (which only `dreamer.py` itself pulls in via `envs/wrappers.py`).

**Design notes.** Each scenario is trained end-to-end and its full
artifact bundle (learning curves, eval curve, video, checkpoint, stage
summary) is written to disk **before** the next scenario starts — same
Colab-preemption-safe pattern as the PPO/DQN notebooks.

**Colab baseline.** Defaults target an **L4 GPU + high-memory runtime**:
single env (the upstream port runs single-threaded), `batch_size=16`,
`batch_length=64`, `train_ratio=512`, `imag_horizon=15` (preset) or
`20` (deadly_corridor / deathmatch — longer-horizon planning).

## 1. Setup

In [ ]:
# --- Environment setup ---
# Works both locally and on Google Colab: `setup_colab` clones + installs the
# repo when a Colab runtime is detected and is a no-op otherwise, so there is
# nothing to uncomment or edit. Locally, run `pip install -e ".[notebooks]"`
# once beforehand.
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from rl_doom.utils import setup_colab
setup_colab(extras="notebooks,dreamer")

import numpy as np
import torch

from rl_doom.paths import load_yaml_config, new_run_dir, write_config
from rl_doom.sb3_utils import gpu_info
from rl_doom.agents.dreamer import UPSTREAM_PIN, train_dreamer

# L4 benefits noticeably from cuDNN autotuning on fixed-shape CNNs.
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Global RNG seeding for notebook-level reproducibility. ``train_dreamer``
# also calls ``tools.set_seed_everywhere`` per-run via the ``seed=`` kwarg;
# this block covers non-Dreamer randomness (numpy, Python ``random``, torch)
# that the notebook may consume outside the training loop.
import random as _random
GLOBAL_SEED = 42
_random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
GPU_INFO = gpu_info()
print(f"Using device: {DEVICE}")
for k, v in GPU_INFO.items():
    print(f"  {k}: {v}")

In [ ]:
# --- Google Drive persistence (Colab only) ---
# No-op off Colab. On Colab this mounts Drive and symlinks the
# `training_jobs/` and `analysis/` trees into it so runs survive the runtime
# being recycled. Point DRIVE_ROOT wherever you want the artifacts to live.
from rl_doom.utils import setup_google_drive

DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
setup_google_drive(DRIVE_ROOT)


## 2. Clone the DreamerV3 PyTorch port

We `git clone` `NM512/dreamerv3-torch` pinned to `UPSTREAM_PIN` and pass
the checkout path into `train_dreamer`. The wrapper imports `models.py`,
`networks.py`, `tools.py`, `exploration.py` from this clone and re-uses
the embedded `_Dreamer` driver — no upstream `dreamer.py` import, so
no `gym` / `ruamel.yaml` deps.

In [ ]:
import subprocess
from pathlib import Path

PORT_PATH = Path("/content/dreamerv3-torch") if Path("/content").exists() else Path.home() / "dreamerv3-torch"
if not PORT_PATH.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/NM512/dreamerv3-torch", str(PORT_PATH)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(PORT_PATH), "checkout", UPSTREAM_PIN], check=True,
)
head = subprocess.check_output(
    ["git", "-C", str(PORT_PATH), "rev-parse", "HEAD"], text=True,
).strip()
assert head == UPSTREAM_PIN, f"port HEAD {head} != pin {UPSTREAM_PIN}"
print(f"DreamerV3 port ready at {PORT_PATH} @ {head[:12]}")

## 3. Load per-scenario configs

Each scenario's hyperparameters, env settings, and training budget live
in `configs/dreamer_<scenario>.yaml`. We load all four upfront so the
loop below can iterate without touching the YAML files again.

In [ ]:
SCENARIOS = ["basic", "deadly_corridor", "defend_the_center", "deathmatch"]

CONFIGS = {s: load_yaml_config(f"dreamer_{s}") for s in SCENARIOS}

# Sanity dump of the knobs that will drive training.
for s, cfg in CONFIGS.items():
    hp = cfg["hyperparams"]
    env_cfg = cfg.get("env", {})
    training = cfg["training"]
    print(
        f"{s:20s}  total_timesteps={training['total_timesteps']:>10,}  "
        f"batch={hp['batch_size']}x{hp['batch_length']}  "
        f"imag_horizon={hp['imag_horizon']:>3}  "
        f"train_ratio={hp['train_ratio']}"
    )

## 4. Train each scenario (artifacts written per scenario)

`train_dreamer` does the full end-to-end training for one scenario:
1. Builds a single `DreamerDoomEnv` (RSSM replaces frame stacking, so no
   `FrameStack` here — just resize + frame-skip).
2. Prefills the replay buffer with random rollouts (`config.prefill`).
3. Runs the upstream's train/eval alternation via `tools.simulate(...)`,
   with checkpoints saved to `checkpoints/latest.pt` after each eval pass.
4. On completion: writes `training.npz`, `learning_curves.png`,
   `eval_performance.png`, the final video bundle, `stage_summary.txt`,
   and updates the `latest` pointer.

The loop pulls hyperparams, env settings, seed, and eval/checkpoint
schedule from the YAML. `config.json` records the full merged view so
each run is reproducible from a single file.

In [ ]:
from IPython.display import Video, display

results = {}
for scenario in SCENARIOS:
    cfg = CONFIGS[scenario]
    hp = dict(cfg["hyperparams"])
    env_cfg = dict(cfg.get("env", {}))
    training_cfg = cfg["training"]
    eval_cfg = cfg.get("eval", {})
    seed = int(cfg.get("seed", 42))
    total_ts = int(training_cfg["total_timesteps"])

    print("\n" + "=" * 70)
    print(f"[Dreamer] {scenario}  |  total_timesteps={total_ts:,}  |  seed={seed}")
    print("=" * 70)

    run_dir = new_run_dir(scenario, "dreamer", seed=seed)
    write_config(
        run_dir,
        env=scenario,
        algo="dreamer",
        seed=seed,
        hyperparams={**hp, "total_timesteps": total_ts},
        env_settings=env_cfg,
        gpu_setup=GPU_INFO,
        training_schedule={
            "checkpoint_freq": int(training_cfg.get("checkpoint_freq", 50_000)),
            "eval_freq": int(eval_cfg.get("eval_freq", 10_000)),
            "eval_episodes": int(eval_cfg.get("n_episodes", 5)),
        },
        upstream_pin=UPSTREAM_PIN,
    )

    def _show(rd, scenario=scenario):
        # _record_dreamer_video writes one mp4 per playthrough; list them all
        # and inline-display the first to keep notebook output manageable.
        vids = sorted((rd / "media").glob(f"dreamer_{scenario}_ep*.mp4"))
        if not vids:
            vids = sorted((rd / "media").glob(f"dreamer_{scenario}_ep*.gif"))
        if vids:
            for v in vids:
                print(f"[video] {v}")
            try:
                display(Video(str(vids[0]), embed=True))
            except Exception as exc:
                print(f"  (inline display skipped: {exc})")

    result = train_dreamer(
        scenario=scenario,
        run_dir=run_dir,
        total_timesteps=total_ts,
        hyperparams=hp,
        env_cfg=env_cfg,
        eval_cfg=eval_cfg,
        training_cfg=training_cfg,
        port_path=PORT_PATH,
        seed=seed,
        device=DEVICE,
        record_video=True,
        video_episodes=int(eval_cfg.get("n_episodes", 5)),
        on_complete=_show,
    )
    results[scenario] = result
    print(
        f"[done] {scenario}: wall={result['wall_time_seconds']:.1f}s | "
        f"fps={result['fps']:.0f} | eval_mean={result['mean_eval_reward']}"
    )

print("\nAll Dreamer scenarios complete.")
for s, r in results.items():
    print(f"  - {s}: {r['run_dir']}")

## 5. Cross-scenario eval summary

Each run already wrote its own `learning_curves.png` and `eval_performance.png`
to `run_dir/figures/`. The cell below just reloads the per-run
`training.npz` files to compare evaluation curves on one axis.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure(figsize=(10, 5))
for scenario, result in results.items():
    npz_path = Path(result["run_dir"]) / "metrics" / "training.npz"
    if not npz_path.exists():
        continue
    data = np.load(npz_path)
    eval_log = data["eval_rewards"]
    if eval_log.ndim == 2 and eval_log.shape[0] > 0:
        steps = eval_log[:, 0]
        means = eval_log[:, 1]
        stds = eval_log[:, 2]
        plt.plot(steps, means, marker="o", label=scenario)
        plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
plt.xlabel("Environment Steps")
plt.ylabel("Eval Reward")
plt.title("Dreamer \u2014 Cross-scenario eval")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Disconnect runtime

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()